# Phase 4: Hybrid Retrieval

This notebook:
- Tests BM25 search (keyword matching)
- Tests FAISS search (semantic similarity)
- Implements hybrid search with RRF fusion
- Compares retrieval methods

## 4.1 Setup

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import time

from src.retrieval import HybridRetriever
from src.query_understanding import QueryUnderstanding

print("✓ Imports successful")

✓ Imports successful


## 4.2 Load Components

In [2]:
# Initialize retriever (loads indices)
retriever = HybridRetriever(indices_dir='../data/indices')

Loading BM25 from ../data/indices/bm25_index.pkl...
✓ BM25 loaded: 100,000 documents
Loading FAISS from ../data/indices/faiss_index.bin...
✓ FAISS loaded: 100,000 vectors
Loading embedding model...
✓ HybridRetriever initialized
  Products: 100,000


In [3]:
# Initialize query understanding
qu = QueryUnderstanding(use_cache=True)
print(f"✓ QueryUnderstanding initialized (cache: {len(qu.cache)} entries)")

✓ QueryUnderstanding initialized (cache: 51 entries)


In [4]:
# Load product data for display
df_products = pd.read_parquet('../data/processed/products.parquet')

# Filter to only products in our index
indexed_ids = set(retriever.product_ids)
df_products = df_products[df_products['product_id'].isin(indexed_ids)].reset_index(drop=True)

# Create lookup dict
product_lookup = df_products.set_index('product_id').to_dict('index')

print(f"✓ Loaded {len(df_products):,} products")

✓ Loaded 100,000 products


## 4.3 Test Individual Search Methods

In [5]:
def display_results(results, title="Results"):
    """Display search results with product details."""
    print(f"\n{title}")
    print("-" * 60)
    for i, r in enumerate(results[:5]):
        product = product_lookup.get(r['product_id'], {})
        product_title = product.get('product_title', 'N/A')[:55]
        sources = r.get('sources', [r.get('source', 'unknown')])
        if isinstance(sources, list):
            sources = ', '.join(sources)
        print(f"{i+1}. [{sources}] {product_title}...")
        print(f"   Score: {r['score']:.4f} | ID: {r['product_id']}")

In [6]:
# Test query
query = "ceramic mugs bulk"
print(f"Query: '{query}'")

# BM25 search
bm25_results = retriever.search(query, method="bm25", top_k=5)
display_results(bm25_results, "BM25 (Keyword) Results")

# FAISS search
faiss_results = retriever.search(query, method="faiss", top_k=5)
display_results(faiss_results, "FAISS (Semantic) Results")

# Hybrid search
hybrid_results = retriever.search(query, method="hybrid", top_k=5)
display_results(hybrid_results, "Hybrid (RRF) Results")

Query: 'ceramic mugs bulk'

BM25 (Keyword) Results
------------------------------------------------------------
1. [bm25] Cute Marshmallow Shaped Hot Chocolate Mugs-Ceramic-Set ...
   Score: 19.3403 | ID: B00GMRJKUG
2. [bm25] when Pappy goes to sleep He counts Tractors 11-Ounce Ce...
   Score: 18.9901 | ID: B07P6ZW3QM
3. [bm25] Serami 15oz White Funnel Ceramic Tall Coffee Mugs with ...
   Score: 18.3779 | ID: B07D6XNJYM
4. [bm25] Bico Ceramic Red & Blue Christmas Gnome 15oz Mugs Set, ...
   Score: 17.8400 | ID: B07X439658
5. [bm25] Don't Be Afraid For I Am With You Mugs, 11OZ/15OZ ceram...
   Score: 17.0399 | ID: B0982QFBVN

FAISS (Semantic) Results
------------------------------------------------------------
1. [faiss] Simple Ceramic Straight Cup, Ceramic Mug Without Handle...
   Score: 0.7341 | ID: B095JTSGSD
2. [faiss] Amici Home Morganite Pink/White 20 oz Ceramic Coffee Mu...
   Score: 0.7074 | ID: B07N6ZJWXJ
3. [faiss] Amici Home Onyx Black/White 20 oz Ceramic Coffee Mugs, ...
   

## 4.4 Compare Methods on Multiple Queries

In [7]:
test_queries = [
    # Exact match queries (BM25 should do well)
    "nike running shoes",
    "iphone 12 case",
    
    # Semantic queries (FAISS should do well)
    "coffee cups",           # synonym for mugs
    "eco friendly bags",     # concept-based
    
    # Wholesale queries
    "candles wholesale lavender",
    "tote bags bulk pack"
]

for query in test_queries:
    print(f"\n{'='*70}")
    print(f"Query: '{query}'")
    print('='*70)
    
    # Get results from all methods
    bm25 = retriever.search(query, method="bm25", top_k=3)
    faiss = retriever.search(query, method="faiss", top_k=3)
    hybrid = retriever.search(query, method="hybrid", top_k=3)
    
    print("\nBM25 Top 3:")
    for r in bm25:
        title = product_lookup.get(r['product_id'], {}).get('product_title', 'N/A')[:50]
        print(f"  - {title}...")
    
    print("\nFAISS Top 3:")
    for r in faiss:
        title = product_lookup.get(r['product_id'], {}).get('product_title', 'N/A')[:50]
        print(f"  - {title}...")
    
    print("\nHybrid Top 3:")
    for r in hybrid:
        title = product_lookup.get(r['product_id'], {}).get('product_title', 'N/A')[:50]
        sources = ', '.join(r['sources'])
        print(f"  - [{sources}] {title}...")


Query: 'nike running shoes'

BM25 Top 3:
  - Nike Women's Renew Run Running Shoes (Black/Pink/O...
  - Nike Womens Air Zoom Pegasus 37 Casual Running Sho...
  - Nike Womens Metcon 4 XD X Running Trainers BV2052 ...

FAISS Top 3:
  - Nike Women's Running Shoes, Black/Gunsmoke/Oil Gre...
  - Nike Kids' Preschool Flex Runner Running Shoes (2....
  - Nike Womens Roshe One Running Shoes (6.5 B(M) US)(...

Hybrid Top 3:
  - [bm25, faiss] Nike Kids' Preschool Flex Runner Running Shoes (2....
  - [bm25, faiss] Nike Women's Running Shoes, Pink Rust Pink Summit ...
  - [bm25, faiss] Nike Womens Air Max Torch 4 Running Shoes Black/Si...

Query: 'iphone 12 case'

BM25 Top 3:
  - Wallme Samsung Galaxy A10E Case with Tempered Glas...
  - Apple Teachers Gift Personalized Apple iPhone Blac...
  - De-Bin Holster Designed for iPhone 12 Pro Max, 11 ...

FAISS Top 3:
  - iPhone 11 Pro Max Case, Henpone Women Girls Cover ...
  - OTTERBOX SYMMETRY SERIES Case for iPhone 11 Pro - ...
  - iPhone 11 Pro Case 

## 4.5 Integrate Query Understanding

In [8]:
def search_with_query_expansion(query: str, retriever, qu, top_k: int = 10):
    """
    Search with query understanding and expansion.
    
    1. Analyze query to extract intent and expanded terms
    2. Build expanded query
    3. Run hybrid search
    """
    # Analyze query
    analysis = qu.analyze(query)
    
    # Build expanded query
    expanded_query = qu.get_expanded_query(query)
    
    # Search with expanded query
    results = retriever.search(expanded_query, method="hybrid", top_k=top_k)
    
    return {
        'original_query': query,
        'analysis': analysis,
        'expanded_query': expanded_query,
        'results': results
    }

In [9]:
# Test with query expansion
query = "ceramic mugs bulk"

print(f"Original Query: '{query}'")
print("-" * 50)

result = search_with_query_expansion(query, retriever, qu, top_k=5)

print(f"\nQuery Analysis:")
print(f"  Product type: {result['analysis']['product_type']}")
print(f"  Attributes: {result['analysis']['attributes']}")
print(f"  Intent: {result['analysis']['intent']}")
print(f"  Expanded terms: {result['analysis']['expanded_terms']}")

print(f"\nExpanded Query: '{result['expanded_query']}'")

print(f"\nSearch Results:")
for i, r in enumerate(result['results']):
    title = product_lookup.get(r['product_id'], {}).get('product_title', 'N/A')[:55]
    sources = ', '.join(r['sources'])
    print(f"{i+1}. [{sources}] {title}...")

Original Query: 'ceramic mugs bulk'
--------------------------------------------------

Query Analysis:
  Product type: mugs
  Attributes: ['ceramic']
  Intent: wholesale_purchase
  Expanded terms: ['porcelain mugs', 'white mugs', 'custom mugs', 'ceramic coffee mugs', 'tableware']

Expanded Query: 'ceramic mugs bulk porcelain mugs white mugs custom mugs'

Search Results:
1. [bm25, faiss] Custom Mugs with Pictures and Words Personalized Adding...
2. [bm25, faiss] Serami 15oz White Funnel Ceramic Tall Coffee Mugs with ...
3. [bm25, faiss] Set of 6 Novelty "Diner" Mugs with Handle - Ceramic, Mu...
4. [bm25, faiss] Customized Ceramic Coffee Mugs with Personalized Text a...
5. [bm25, faiss] Amici Home Morganite Pink/White 20 oz Ceramic Coffee Mu...


## 4.6 Compare: With vs Without Query Expansion

In [10]:
comparison_queries = [
    "coffee cups wholesale",
    "eco friendly shopping bags",
    "organic skincare products bulk"
]

for query in comparison_queries:
    print(f"\n{'='*70}")
    print(f"Query: '{query}'")
    print('='*70)
    
    # Without expansion
    results_no_exp = retriever.search(query, method="hybrid", top_k=3)
    
    # With expansion
    expanded = qu.get_expanded_query(query)
    results_with_exp = retriever.search(expanded, method="hybrid", top_k=3)
    
    print(f"\nExpanded: '{expanded}'")
    
    print("\nWithout Expansion:")
    for r in results_no_exp:
        title = product_lookup.get(r['product_id'], {}).get('product_title', 'N/A')[:50]
        print(f"  - {title}...")
    
    print("\nWith Expansion:")
    for r in results_with_exp:
        title = product_lookup.get(r['product_id'], {}).get('product_title', 'N/A')[:50]
        print(f"  - {title}...")


Query: 'coffee cups wholesale'

Expanded: 'coffee cups wholesale mugs beverage containers coffee mugs'

Without Expansion:
  - 500-CT Disposable Gray 12-OZ Hot Beverage Cups wit...
  - Trader Joe's Coffee Cups - Single Serve - Medium R...
  - [100 Pack] 12 oz. White Paper Hot Coffee Cups...

With Expansion:
  - Serami 15oz White Funnel Ceramic Tall Coffee Mugs ...
  - Woffit Wine Glass Storage - Set of 2 Quilted Packi...
  - STRATA CUPS Camera Lens Coffee Mug -13.5oz, SUPER ...

Query: 'eco friendly shopping bags'

Expanded: 'eco friendly shopping bags reusable bags sustainable bags cloth bags'

Without Expansion:
  - Reusable Produce Bags 15 Pcs, Barcode Scannable, W...
  - Reusable Produce Bags Cotton Washable with Tare We...
  - BeeGreen Grocery Bags Reusable Foldable 6 Pack Sho...

With Expansion:
  - Reusable Produce Bags Cotton Washable with Tare We...
  - Reusable Shopping Bags 5 Pack Reusable Tote Bags M...
  - BeeGreen Grocery Bags Reusable Foldable 6 Pack Sho...

Query: 'org

## 4.7 Latency Analysis

In [11]:
query = "ceramic mugs wholesale"
n_runs = 10

# Measure BM25
times_bm25 = []
for _ in range(n_runs):
    start = time.time()
    retriever.search(query, method="bm25", top_k=20)
    times_bm25.append((time.time() - start) * 1000)

# Measure FAISS
times_faiss = []
for _ in range(n_runs):
    start = time.time()
    retriever.search(query, method="faiss", top_k=20)
    times_faiss.append((time.time() - start) * 1000)

# Measure Hybrid
times_hybrid = []
for _ in range(n_runs):
    start = time.time()
    retriever.search(query, method="hybrid", top_k=20)
    times_hybrid.append((time.time() - start) * 1000)

print("Latency Analysis (ms):")
print("-" * 40)
print(f"BM25:   mean={np.mean(times_bm25):.1f}, p95={np.percentile(times_bm25, 95):.1f}")
print(f"FAISS:  mean={np.mean(times_faiss):.1f}, p95={np.percentile(times_faiss, 95):.1f}")
print(f"Hybrid: mean={np.mean(times_hybrid):.1f}, p95={np.percentile(times_hybrid, 95):.1f}")

Latency Analysis (ms):
----------------------------------------
BM25:   mean=54.9, p95=56.0
FAISS:  mean=9.2, p95=11.0
Hybrid: mean=65.7, p95=66.2


## 4.8 Summary

In [12]:
print("\n" + "=" * 60)
print("PHASE 4 COMPLETE")
print("=" * 60)

print("""
Hybrid Retrieval Summary
------------------------

Components:
  - BM25: Keyword matching (exact terms, brands)
  - FAISS: Semantic similarity (synonyms, concepts)
  - RRF Fusion: Combines rankings from both

Pipeline:
  Query -> Query Understanding -> Expansion -> Hybrid Search -> Results

Files:
  - src/retrieval.py (HybridRetriever class)
  - src/query_understanding.py (QueryUnderstanding class)

Next: Run 05_reranking.ipynb (Train reranker for better relevance)
""")


PHASE 4 COMPLETE

Hybrid Retrieval Summary
------------------------

Components:
  - BM25: Keyword matching (exact terms, brands)
  - FAISS: Semantic similarity (synonyms, concepts)
  - RRF Fusion: Combines rankings from both

Pipeline:
  Query -> Query Understanding -> Expansion -> Hybrid Search -> Results

Files:
  - src/retrieval.py (HybridRetriever class)
  - src/query_understanding.py (QueryUnderstanding class)

Next: Run 05_reranking.ipynb (Train reranker for better relevance)

